In [1]:
import os
import sys
import pandas as pd
import numpy as np
from scipy import stats


spend = pd.DataFrame({
    'customer_id': [f'C{str(i).zfill(3)}' for i in range(1,21)],
    'product': ['Standard']*10 + ['Premium']*10,
    'monthly_spend': [420,390,510,470,380,430,460,490,410,440,
                      980,1150,870,1320,1050,1200,940,1080,1410,990],
    'credit_score':  [680,710,690,720,660,700,715,695,705,685,
                      760,790,750,810,775,800,765,780,820,755],
    'age': [28,34,45,30,52,41,37,29,48,33,44,38,55,31,47,36,42,50,29,39]
})

loans = pd.DataFrame({
    'app_id': [f'L{str(i).zfill(3)}' for i in range(1,11)],
    'region': ['North']*5 + ['South']*5,
    'loan_amount': [15000,22000,8000,18000,12000,30000,25000,9000,27000,11000],
    'approved': [1,1,0,1,0,1,1,0,1,0],
    'processing_days': [3,5,7,4,6,4,3,8,5,9]
})

In [2]:
print(spend["monthly_spend"].mean())
print(spend["monthly_spend"].median())
print(spend["monthly_spend"].std())

769.5
690.0
359.27522070207726


In [3]:
print(spend.describe())
range_1 = spend["credit_score"].max() - spend["credit_score"].min()
print("=========")
print(range_1)
print("=========")
sorted_1 = spend.sort_values("credit_score",ascending = False)
print(sorted_1[0:1])

       monthly_spend  credit_score        age
count      20.000000     20.000000  20.000000
mean      769.500000    738.250000  39.400000
std       359.275221     48.020692   8.280605
min       380.000000    660.000000  28.000000
25%       437.500000    698.750000  32.500000
50%       690.000000    735.000000  38.500000
75%      1057.500000    776.250000  45.500000
max      1410.000000    820.000000  55.000000
160
   customer_id  product  monthly_spend  credit_score  age
18        C019  Premium           1410           820   29


In [4]:
premium = spend[spend["product"] == "Premium"]
standard = spend[spend["product"] == "Standard"]

print(premium.head(2))
print(standard.head(2))

   customer_id  product  monthly_spend  credit_score  age
10        C011  Premium            980           760   44
11        C012  Premium           1150           790   38
  customer_id   product  monthly_spend  credit_score  age
0        C001  Standard            420           680   28
1        C002  Standard            390           710   34


In [5]:
variation = spend.groupby("product")["monthly_spend"].agg(average = 'mean', deviation ='std')
variation["cv"] = variation["deviation"]/variation["average"]
print(variation)

          average   deviation        cv
product                                
Premium    1099.0  171.558218  0.156104
Standard    440.0   42.426407  0.096424


In [6]:
premium = spend[spend["product"] == "Premium"]["monthly_spend"]
standard = spend[spend["product"] == "Standard"]["monthly_spend"]

t_stat, pval = stats.ttest_ind(standard, premium)
print(f"t= {t_stat:.2f}, p = {pval:.2f}")

if pval <0.05:
    print("Reject Null Hypothesis: There is a significant statistical difference in null standard and premium")

t= -11.79, p = 0.00
Reject Null Hypothesis: There is a significant statistical difference in null standard and premium


In [7]:
north = loans[loans["region"]== "North"]["processing_days"]
south = loans[loans["region"]== "South"]["processing_days"]

avg_north = north.mean()
avg_south = south.mean()

print(avg_north)
print(avg_south)

tstat, p_val = stats.ttest_ind(north, south)

print(f"t = {tstat:.2f}, p = {p_val:.2f}")

if p_val<0.05:
    print("Reject Null Hypothesis: There is a significant difference in processing days")
else:
    print("Fail to reject Null Hypothesis: There is no significant difference in processing days")

5.0
5.8
t = -0.59, p = 0.57
Fail to reject Null Hypothesis: There is no significant difference in processing days


In [8]:
spend.to_csv(r"C:\Users\chest\OneDrive\Desktop\Capital One\spend.csv",index = False, encoding = "utf-8")

In [9]:
loans.to_csv(r"C:\Users\chest\OneDrive\Desktop\Capital One\loans.csv",index = False, encoding = "utf-8")

In [10]:
spend.describe()

,monthly_spend,credit_score,age
count,20.000000,20.000000,20.000000
mean,769.500000,738.250000,39.400000
std,359.275221,48.020692,8.280605
min,380.000000,660.000000,28.000000
25%,437.500000,698.750000,32.500000
50%,690.000000,735.000000,38.500000
75%,1057.500000,776.250000,45.500000
max,1410.000000,820.000000,55.000000


In [11]:
Q1 = spend["monthly_spend"].quantile(0.25)
Q3 = spend["monthly_spend"].quantile(0.75)

IQR =  Q3-Q1

print(Q1)
print(Q3)
print(IQR)

IQR_low = Q1  - 1.5*IQR
IQR_high = Q3 + 1.5*IQR

print(IQR_low,IQR_high)

437.5
1057.5
620.0
-492.5 1987.5


In [12]:
standard_scores = spend[spend["product"]=="Standard"]["credit_score"]
print(f"standard score for population: {standard_scores.mean()}")

standard score for population: 696.0


In [13]:
ttest, ptwo = stats.ttest_1samp(standard_scores, popmean= 700)
pone = ptwo/2

print(ttest,pone)

-0.6998542122237651 0.25085292030908324


In [14]:
loan_north = loans[loans["region"]=="North"]["loan_amount"]
loan_south  =loans[loans["region"]=="South"]["loan_amount"]

print(loan_north.head())

ttest_1, pval1 = stats.ttest_ind(loan_north,loan_south)

print(ttest_1, pval1)

if pval1 < 0.05:
    print("Reject null hypothesis: Loan amounts are not the same/similar")
else:
    print("Fail to reject null: Loan amounts are same")

0    15000
1    22000
2     8000
3    18000
4    12000
Name: loan_amount, dtype: int64
-1.0896313215662032 0.30760613879149823
Fail to reject null: Loan amounts are same


In [15]:
for region in ['North','South']:
    group = loans[loans['region']==region]
    rate = group['approved'].mean()
    print(f"{region}: approval rate {rate:.0%}, avg loan ${group['loan_amount'].mean():,.0f}")

North: approval rate 60%, avg loan $15,000
South: approval rate 60%, avg loan $20,400


In [16]:

txns = pd.DataFrame({
    'txn_id':      [f'T{str(i).zfill(3)}' for i in range(1,26)],
    'customer_id': ['C01','C02','C03','C01','C04','C02','C05','C03','C04','C01',
                    'C05','C02','C03','C04','C01','C05','C02','C03','C04','C05',
                    'C01','C02','C03','C04','C05'],
    'txn_date':    pd.to_datetime([
                    '2024-01-05','2024-01-08','2024-01-12','2024-01-15','2024-01-18',
                    '2024-01-22','2024-02-01','2024-02-05','2024-02-09','2024-02-14',
                    '2024-02-18','2024-02-22','2024-03-01','2024-03-05','2024-03-10',
                    '2024-03-12','2024-03-15','2024-03-18','2024-03-20','2024-03-25',
                    '2024-03-27','2024-03-28','2024-03-29','2024-03-30','2024-03-31']),
    'category':    ['Dining','Travel','Groceries','Travel','Dining','Groceries',
                    'Shopping','Dining','Travel','Shopping','Groceries','Shopping',
                    'Travel','Dining','Groceries','Dining','Travel','Shopping',
                    'Groceries','Shopping','Dining','Groceries','Dining','Shopping','Travel'],
    'amount':      [42.50,310.00,88.75,520.00,67.20,54.30,199.99,35.00,875.00,
                    120.00,73.40,245.00,640.00,91.00,62.10,48.50,430.00,310.00,
                    44.80,560.00,29.90,97.60,55.00,188.00,720.00],
    'channel':     ['Online','In-store','Online','Online','In-store','In-store',
                    'Online','In-store','Online','In-store','Online','Online',
                    'In-store','Online','In-store','In-store','Online','Online',
                    'In-store','Online','Online','In-store','Online','Online','In-store'],
    'status':      ['Completed','Completed','Completed','Completed','Declined',
                    'Completed','Completed','Completed','Completed','Completed',
                    'Completed','Completed','Completed','Completed','Completed',
                    'Declined','Completed','Completed','Completed','Completed',
                    'Completed','Completed','Completed','Completed','Completed']
})

customers = pd.DataFrame({
    'customer_id':  ['C01','C02','C03','C04','C05'],
    'name':         ['Alice Martin','Bob Chen','Carol Davis','David Kim','Eva Lopez'],
    'segment':      ['Premium','Standard','Premium','Standard','Premium'],
    'join_date':    pd.to_datetime(['2021-03-15','2022-07-01','2020-11-20','2023-01-10','2021-09-05']),
    'state':        ['TX','CA','NY','TX','FL'],
    'credit_limit': [10000, 5000, 12000, 4500, 15000]
})

disputes = pd.DataFrame({
    'dispute_id':      ['D001','D002','D003','D004','D005'],
    'txn_id':          ['T004','T009','T012','T017','T020'],
    'reason':          ['Unauthorized','Duplicate charge','Item not received',
                        'Unauthorized','Item not received'],
    'resolved':        [1,1,0,1,0],
    'resolution_days': [5.0, 3.0, None, 7.0, None]
})

In [25]:
for col in txns.columns:
    if txns[col].isnull().sum() != 0:
        txns[col].fillna()

print(txns.isnull().sum())

txn_id         0
customer_id    0
txn_date       0
category       0
amount         0
channel        0
status         0
dtype: int64


In [33]:
txns.info()
txns.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   txn_id       25 non-null     object        
 1   customer_id  25 non-null     object        
 2   txn_date     25 non-null     datetime64[ns]
 3   category     25 non-null     object        
 4   amount       25 non-null     float64       
 5   channel      25 non-null     object        
 6   status       25 non-null     object        
dtypes: datetime64[ns](1), float64(1), object(5)
memory usage: 1.5+ KB


,txn_id,customer_id,txn_date,category,amount,channel,status
0,T001,C01,2024-01-05,Dining,42.50,Online,Completed
1,T002,C02,2024-01-08,Travel,310.00,In-store,Completed
2,T003,C03,2024-01-12,Groceries,88.75,Online,Completed
3,T004,C01,2024-01-15,Travel,520.00,Online,Completed
4,T005,C04,2024-01-18,Dining,67.20,In-store,Declined


In [39]:
completed_trans = txns[(txns["status"]=="Completed") & (txns["amount"]>200)]
completed_trans.head()

,txn_id,customer_id,txn_date,category,amount,channel,status
1,T002,C02,2024-01-08,Travel,310.0,In-store,Completed
3,T004,C01,2024-01-15,Travel,520.0,Online,Completed
8,T009,C04,2024-02-09,Travel,875.0,Online,Completed
11,T012,C02,2024-02-22,Shopping,245.0,Online,Completed
12,T013,C03,2024-03-01,Travel,640.0,In-store,Completed


In [47]:
print(len(completed_trans))

9


In [55]:
txns["month"] = txns["txn_date"].dt.month
txns["month_name"]= txns["txn_date"].dt.month_name()
txns.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   txn_id       25 non-null     object        
 1   customer_id  25 non-null     object        
 2   txn_date     25 non-null     datetime64[ns]
 3   category     25 non-null     object        
 4   amount       25 non-null     float64       
 5   channel      25 non-null     object        
 6   status       25 non-null     object        
 7   month        25 non-null     int32         
 8   month_name   25 non-null     object        
dtypes: datetime64[ns](1), float64(1), int32(1), object(6)
memory usage: 1.8+ KB


In [66]:
transact_month =  txns.groupby("month_name")["amount"].agg(cnt = "count", total = "sum")
transact_month.sort_values("cnt",ascending=False, inplace= True)
print(transact_month.head())

            cnt    total
month_name              
March        13  3276.90
February      6  1548.39
January       6  1082.75


In [69]:
spend_category = txns.groupby("category")["amount"].agg(total = "sum",average = "mean",cnt = "count")
print(spend_category)

             total     average  cnt
category                           
Dining      369.10   52.728571    7
Groceries   420.95   70.158333    6
Shopping   1622.99  270.498333    6
Travel     3495.00  582.500000    6


In [74]:
customers[["first_name", "last_name"]] = customers["name"].str.split(' ',expand=True)

In [75]:
customers.head()

,customer_id,name,segment,join_date,state,credit_limit,first_name,last_name
0,C01,Alice Martin,Premium,2021-03-15,TX,10000,Alice,Martin
1,C02,Bob Chen,Standard,2022-07-01,CA,5000,Bob,Chen
2,C03,Carol Davis,Premium,2020-11-20,NY,12000,Carol,Davis
3,C04,David Kim,Standard,2023-01-10,TX,4500,David,Kim
4,C05,Eva Lopez,Premium,2021-09-05,FL,15000,Eva,Lopez


In [76]:
txns.to_csv(r"C:\Users\chest\OneDrive\Desktop\Capital One\txns.csv",index = False, encoding = "utf-8")

In [77]:
customers.to_csv(r"C:\Users\chest\OneDrive\Desktop\Capital One\customers.csv",index = False, encoding = "utf-8")

In [78]:
disputes.to_csv(r"C:\Users\chest\OneDrive\Desktop\Capital One\disputes.csv",index = False, encoding = "utf-8")

In [79]:
txns["spending_tier"] = pd.cut(txns["amount"], bins=[0,100,500, float('inf')], labels=['Low','Medium','High'], right=False)

print(txns['spending_tier'].value_counts().sort_index())

spending_tier
Low       13
Medium     7
High       5
Name: count, dtype: int64


In [80]:
disputes.isnull().sum()

dispute_id         0
txn_id             0
reason             0
resolved           0
resolution_days    2
dtype: int64

In [81]:
for col in disputes:
    if disputes[col].isnull().sum()!=0:
        disputes[col].fillna("Pending")